# 🧬 Project 5 — Semantic Correspondence
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data
# Scarichiamo anche PF-Pascal per lo Stage 4
!mkdir -p ./data/PF-Pascal
!wget -O ./data/PF-Pascal/pfpascal.zip https:// some-url-for-pfpascal (o comando manuale)

import os, torch, gc
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

## 🔍 1. Evaluation Baseline (Multi-Backbone)

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --batch_size 1 --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

## 🚀 2. Training Stage

In [ ]:
CKPT_PATH = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
if not os.path.exists(CKPT_PATH):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir "$DRIVE_CKPTS/lora_only"
else: print(f'[OK] Checkpoint trovato: {CKPT_PATH}')

## 🌍 4. Generalizzazione e Robustezza (Stage 4)

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
print(f'Test su PF-Pascal usando: {CKPT_LORA}')
!python evaluate.py --dataset_root ./data/PF-Pascal --dataset_type pfpascal --checkpoint "$CKPT_LORA" --results_file /content/drive/MyDrive/AML/Results/gen_pfpascal.txt

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
print(f'Test Robustezza Geometrica usando: {CKPT_LORA}')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$CKPT_LORA" --results_file /content/drive/MyDrive/AML/Results/robustness_rot.txt

## ⚖️ 5. Demo Interattive

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')